# ทดสอบ Custom Code (Fix + FP16 + tile=800)

In [ ]:
# @title 1. ติดตั้ง
!pip install opencv-python torch torchvision --quiet
print("Done")

In [ ]:
# @title 2. อัปโหลดรูป
from google.colab import files
uploaded = files.upload()
input_path = list(uploaded.keys())[0] if uploaded else None
print(f"Input: {input_path}")

In [ ]:
# @title 3. Custom code + รัน x2plus
import os, urllib.request, time
from collections import OrderedDict

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ResidualDenseBlock(nn.Module):
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.conv1 = nn.Conv2d(nf, gc, 3, 1, 1)
        self.conv2 = nn.Conv2d(nf + gc, gc, 3, 1, 1)
        self.conv3 = nn.Conv2d(nf + 2 * gc, gc, 3, 1, 1)
        self.conv4 = nn.Conv2d(nf + 3 * gc, gc, 3, 1, 1)
        self.conv5 = nn.Conv2d(nf + 4 * gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(nf, gc)
        self.rdb2 = ResidualDenseBlock(nf, gc)
        self.rdb3 = ResidualDenseBlock(nf, gc)
    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out * 0.2 + x

class RRDBNet(nn.Module):
    def __init__(self, in_nc=3, out_nc=3, nf=64, nb=23, gc=32, scale=4):
        super().__init__()
        self.scale = scale
        pu = 2 if scale == 2 else 1
        self.conv_first = nn.Conv2d(in_nc * pu * pu, nf, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(nf, gc) for _ in range(nb)])
        self.conv_body = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_hr = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_last = nn.Conv2d(nf, out_nc, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    def forward(self, x):
        if self.scale == 2:
            feat = F.pixel_unshuffle(x, 2)
        else:
            feat = x
        feat = self.conv_first(feat)
        body_feat = self.conv_body(self.body(feat))
        feat = feat + body_feat
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode='nearest')))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode='nearest')))
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out

def _remap_ultrasharp_key(k):
    inner = k[6:]
    if inner.startswith("0."): return "conv_first." + inner[2:]
    if inner.startswith("1.sub."):
        sub_inner = inner[6:]; dot = sub_inner.find(".")
        block_idx = int(sub_inner[:dot]); rest = sub_inner[dot+1:]
        if block_idx < 23:
            rest = rest.replace("RDB","rdb").replace(".0","")
            return f"body.{block_idx}.{rest}"
        if block_idx == 23: return f"conv_body.{rest}"
        return None
    if inner.startswith("3."): return "conv_up1." + inner[2:]
    if inner.startswith("6."): return "conv_up2." + inner[2:]
    if inner.startswith("8."): return "conv_hr." + inner[2:]
    if inner.startswith("10."): return "conv_last." + inner[3:]
    return None

def load_model(model_path, device):
    ckpt = torch.load(model_path, map_location=device, weights_only=True)
    for k in ["params","params_ema","state_dict"]:
        if k in ckpt: state = ckpt[k]; break
    else: state = ckpt
    first_key = next(iter(state.keys()))
    if first_key.startswith("model."):
        remapped = OrderedDict()
        for k, v in state.items():
            if k.startswith("module."): k = k[7:]
            mapped = _remap_ultrasharp_key(k)
            if mapped is not None: remapped[mapped] = v
        state = remapped
    for key in state:
        if "conv_first.weight" in key:
            ckpt_in_nc = state[key].shape[1]; break
    else: ckpt_in_nc = 3
    scale = 2 if ckpt_in_nc >= 6 else 4
    in_nc = ckpt_in_nc // 4 if scale == 2 else ckpt_in_nc
    model = RRDBNet(in_nc=in_nc, scale=scale)
    new_state = OrderedDict()
    for k, v in state.items():
        new_state[k[7:] if k.startswith("module.") else k] = v
    model.load_state_dict(new_state)
    model = model.to(device).eval()
    return model

@torch.no_grad()
def inference_tiled(model, img_bgr, tile_size=400, tile_pad=10, pre_pad=10, half=False):
    h, w = img_bgr.shape[:2]
    img = img_bgr[:, :, ::-1].copy()
    img_t = torch.from_numpy(img).permute(2,0,1).float().unsqueeze(0)/255.0
    device = next(model.parameters()).device
    if half:
        img_t = img_t.half()
    img_t = img_t.to(device)
    scale = model.scale

    if pre_pad > 0:
        img_t = F.pad(img_t, (0, pre_pad, 0, pre_pad), 'reflect')
    mod_scale = scale
    mod_pad_h, mod_pad_w = 0, 0
    _, _, h_p, w_p = img_t.size()
    if h_p % mod_scale != 0:
        mod_pad_h = mod_scale - h_p % mod_scale
    if w_p % mod_scale != 0:
        mod_pad_w = mod_scale - w_p % mod_scale
    if mod_pad_h > 0 or mod_pad_w > 0:
        img_t = F.pad(img_t, (0, mod_pad_w, 0, mod_pad_h), 'reflect')

    _, _, h_full, w_full = img_t.size()
    oh, ow = h_full * scale, w_full * scale
    out = torch.zeros(1, 3, oh, ow, device=device, dtype=torch.half if half else torch.float)

    for y in range(0, h_full, tile_size):
        for x in range(0, w_full, tile_size):
            y0 = max(0, y - tile_pad)
            y1 = min(h_full, y + tile_size + tile_pad)
            x0 = max(0, x - tile_pad)
            x1 = min(w_full, x + tile_size + tile_pad)
            patch = img_t[:, :, y0:y1, x0:x1]
            pred = model(patch)
            dy = min(tile_size, h_full - y) * scale
            dx = min(tile_size, w_full - x) * scale
            out[:, :, y * scale:y * scale + dy, x * scale:x * scale + dx] = pred[:, :, :dy, :dx]

    if mod_pad_h > 0 or mod_pad_w > 0:
        out = out[:, :, :oh - mod_pad_h * scale, :ow - mod_pad_w * scale]
    if pre_pad > 0:
        out = out[:, :, pre_pad * scale:-pre_pad * scale, pre_pad * scale:-pre_pad * scale]

    result = out.squeeze(0).permute(1,2,0).clamp(0,1).cpu().numpy()*255.0
    result = result[:,:,::-1].astype(np.uint8)
    return result

def upscale(image_path, output_path="output.jpg"):
    url = "https://huggingface.co/nateraw/real-esrgan/resolve/main/RealESRGAN_x2plus.pth"
    model_path = "RealESRGAN_x2plus.pth"
    if not os.path.exists(model_path):
        print("Downloading x2plus model...")
        urllib.request.urlretrieve(url, model_path)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_half = device == "cuda"
    model = load_model(model_path, device)
    if use_half:
        model = model.half()
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Cannot read: {image_path}")
    h, w = img.shape[:2]
    print(f"Input: {w}x{h}  |  Device: {device}  |  half={use_half}")
    output = inference_tiled(model, img, tile_size=400, half=use_half)
    oh, ow = output.shape[:2]
    print(f"Output: {ow}x{oh}")
    cv2.imwrite(output_path, output, [cv2.IMWRITE_JPEG_QUALITY, 95])
    print(f"Saved: {output_path}")
    return output_path

# ── รัน ──
if input_path is None:
    raise RuntimeError("No input file uploaded")

print("="*40)
print("Custom x2plus (skip connection + pre_pad + FP16 + tile=800)")
print("="*40)
t0 = time.time()
out = upscale(input_path, "output_custom.jpg")
t = time.time() - t0
print(f"Time: {t:.1f}s")

In [ ]:
# @title 4. Preview (ย่อขนาด) + ดาวน์โหลด
from google.colab.patches import cv2_imshow
from google.colab import files
import cv2

img = cv2.imread("output_custom.jpg")
h, w = img.shape[:2]
scale = 800 / w  # ย่อให้กว้างสุด 800px
thumb = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
cv2_imshow(thumb)
print(f"Preview: {int(w*scale)}x{int(h*scale)}  |  Full: {w}x{h}")
files.download("output_custom.jpg")